# 11 — Handling Class Imbalance

**Difficulty:** ⭐⭐⭐⭐ | **Estimated time:** 60 min

**Week 12 theme:** Airbnb-style listing price prediction — detecting luxury listings

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain why **accuracy is a misleading metric** on imbalanced datasets and which metrics to use instead.
2. Implement **random oversampling** and **random undersampling** from scratch.
3. Implement **SMOTE** (Synthetic Minority Oversampling Technique) from scratch.
4. Apply **class weights** and **threshold tuning** as lightweight zero-resampling alternatives.
5. Know **when NOT to resample** and what to watch out for in production.


## The Analogy: The Useless Fraud Detector

Imagine you are building a fraud detector. In your dataset, 99% of transactions are legitimate and 1% are fraudulent.

A model that **always predicts "legitimate"** achieves **99% accuracy**. Impressive on paper. Completely useless in practice — it never catches a single fraudster.

The model has learned the path of least resistance: ignore the hard minority class and just parrot back the majority.

**SMOTE** breaks this cycle by creating *synthetic* minority-class examples during training. Think of a detective agency running tabletop fraud simulations — fabricated scenarios that look plausible but weren't in the original case files — so that investigators learn to recognise patterns they have rarely seen in the wild.

SMOTE works by finding real minority samples and generating new synthetic samples *between* them in feature space. The model is forced to learn the shape of the minority class rather than ignoring it.


## Why Does This Matter for ML?

Class imbalance is the **default condition** for many high-value ML problems:

| Problem | Minority class | Typical ratio |
|---|---|---|
| Credit card fraud | Fraud | 0.1 – 1% |
| Medical diagnosis | Disease | 1 – 10% |
| Churn prediction | Churned user | 2 – 15% |
| Rare event detection | Event | < 0.1% |
| Luxury listing detection | Luxury | 5% (our task today) |

**Why standard training fails:**
- Cross-entropy / log-loss treats every sample equally. A model minimising loss on 95% negatives and 5% positives will mostly just get the negatives right.
- The decision boundary lands wherever the majority class ends — the minority region is ignored.

**What we actually care about:** catching the minority class. A missed luxury listing is a missed business opportunity. A missed fraud is a financial loss. A missed cancer diagnosis is a life.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.decomposition import PCA

np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Synthetic Airbnb dataset ──────────────────────────────────────────────────
N = 3000

rooms         = np.random.randint(1, 6, size=N).astype(float)
bathrooms     = np.random.choice([1.0, 1.5, 2.0, 2.5, 3.0], size=N)
distance_km   = np.abs(np.random.normal(3, 2, size=N))
review_score  = np.clip(np.random.normal(4.3, 0.4, size=N), 1, 5)
sqft          = np.random.normal(700, 200, size=N).clip(200, 2000)
amenity_score = np.random.beta(2, 5, size=N) * 10  # 0–10, right-skewed

# Continuous price signal
log_price = (
    4.5
    + 0.25 * rooms
    + 0.15 * bathrooms
    - 0.06 * distance_km
    + 0.30 * review_score
    + 0.002 * sqft
    + 0.05 * amenity_score
    + np.random.normal(0, 0.35, size=N)
)

# Binary target: top 5% by log_price = "luxury" (class 1)
threshold = np.percentile(log_price, 95)
is_luxury = (log_price >= threshold).astype(int)

# Feature matrix
FEATURE_NAMES = ['rooms', 'bathrooms', 'distance_km', 'review_score', 'sqft', 'amenity_score']
X = np.column_stack([rooms, bathrooms, distance_km, review_score, sqft, amenity_score])
y = is_luxury

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train / test split — stratified to preserve imbalance ratio
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print('Class distribution:')
vals, counts = np.unique(y_train, return_counts=True)
for v, c in zip(vals, counts):
    label = 'luxury' if v == 1 else 'regular'
    print(f'  Class {v} ({label}): {c} samples ({100*c/len(y_train):.1f}%)')

print(f'\nClass imbalance ratio: {counts[0]/counts[1]:.1f}:1  (majority:minority)')

## Why Accuracy Fails on Imbalanced Data

Consider a test set with 950 regular listings and 50 luxury listings.

| Prediction strategy | Accuracy | Precision (luxury) | Recall (luxury) | F1 (luxury) |
|---|---|---|---|---|
| Always predict "regular" | 95% | — | 0% | 0% |
| Perfect classifier | 100% | 100% | 100% | 100% |

A model predicting "regular" for every single listing scores 95% accuracy — better than most real classifiers — **while having absolutely zero predictive power for the minority class**.

**Better metrics for imbalanced problems:**

- **Precision** = TP / (TP + FP): of the listings we flagged as luxury, what fraction actually are?  
- **Recall** = TP / (TP + FN): of all actual luxury listings, what fraction did we catch?  
- **F1** = harmonic mean of precision and recall — punishes models that sacrifice one for the other.  
- **AUC-ROC**: area under the ROC curve — measures ranking quality across all thresholds. 0.5 = random, 1.0 = perfect.

Rule of thumb: **always report AUC-ROC and F1 for imbalanced problems**. Drop accuracy from your primary dashboard.


In [ ]:
np.random.seed(42)

def evaluate(name, y_true, y_pred, y_prob=None):
    """Print a concise metric summary for one model/strategy."""
    auc = roc_auc_score(y_true, y_prob) if y_prob is not None else float('nan')
    return {
        'Method':    name,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 3),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 3),
        'Recall':    round(recall_score(y_true, y_pred, zero_division=0), 3),
        'F1':        round(f1_score(y_true, y_pred, zero_division=0), 3),
        'AUC-ROC':   round(auc, 3),
    }

# ── Baseline 1: always predict majority class ─────────────────────────────────
y_pred_naive = np.zeros(len(y_test), dtype=int)  # always predict "regular"
metrics_naive = evaluate('Always predict 0', y_test, y_pred_naive)

# ── Baseline 2: standard logistic regression (no resampling, no class weights) ─
lr_base = LogisticRegression(random_state=42, max_iter=500)
lr_base.fit(X_train, y_train)
y_pred_base  = lr_base.predict(X_test)
y_prob_base  = lr_base.predict_proba(X_test)[:, 1]
metrics_base = evaluate('LR (no adjustment)', y_test, y_pred_base, y_prob_base)

# ── Side-by-side confusion matrices ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (preds, title) in zip(axes, [
    (y_pred_naive, 'Always predict \'regular\''),
    (y_pred_base,  'Logistic Regression (baseline)'),
]):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['regular', 'luxury'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)
plt.tight_layout()
plt.show()

print(pd.DataFrame([metrics_naive, metrics_base]).to_string(index=False))
print('\nNote: "Always predict 0" has 0% recall for luxury — it never catches a single listing.')

## Strategy 1: Class Weights

The simplest fix requires **zero changes to your data** — just tell the model to care more about the minority class.

Class weights modify the loss function so that misclassifying a minority-class sample is penalised more heavily:

$$\mathcal{L}_{weighted} = \sum_i w_{y_i} \cdot \text{loss}(\hat{y}_i, y_i)$$

With `class_weight='balanced'`, sklearn automatically sets:

$$w_c = \frac{N}{K \times n_c}$$

where $N$ is total samples, $K$ is number of classes, $n_c$ is samples in class $c$.

For our 95/5 split this roughly means the rare luxury listings get **~19× the weight** of regular listings in the loss. The model can no longer ignore them cheaply.

**Pros:** zero computational cost, no data augmentation, works immediately.  
**Cons:** only adjusts the loss — not the feature space. If features genuinely don't separate the classes, weights alone won't help.


In [ ]:
np.random.seed(42)

# Apply balanced class weights — only one parameter change
lr_weighted = LogisticRegression(class_weight='balanced', random_state=42, max_iter=500)
lr_weighted.fit(X_train, y_train)

y_pred_weighted = lr_weighted.predict(X_test)
y_prob_weighted = lr_weighted.predict_proba(X_test)[:, 1]
metrics_weighted = evaluate('LR (class_weight=balanced)', y_test, y_pred_weighted, y_prob_weighted)

# Show what 'balanced' actually computes
classes, counts = np.unique(y_train, return_counts=True)
N_total = len(y_train)
K = len(classes)
for cls, cnt in zip(classes, counts):
    w = N_total / (K * cnt)
    label = 'luxury' if cls == 1 else 'regular'
    print(f'  Class {cls} ({label:7s}): count={cnt:4d}  weight={w:.2f}')

print()
# Compare baseline vs weighted
print(pd.DataFrame([metrics_base, metrics_weighted]).to_string(index=False))
print()
print(f'Recall improvement  : {metrics_weighted["Recall"] - metrics_base["Recall"]:+.3f}')
print(f'Precision trade-off : {metrics_weighted["Precision"] - metrics_base["Precision"]:+.3f}')
print('(Higher recall usually comes at some cost to precision — this is expected)')

## Strategy 2: Random Oversampling

**Idea:** randomly duplicate minority class samples until the classes are balanced.

Think of photocopying some of the rare case files until you have as many as the common ones. The information content doesn't change — you're just reading the same files more times — but the model gets more gradient signal from the minority class.

**Pros:** simple; guarantees exact balance; works with any model.  
**Cons:** exact duplicates — the model may memorise specific minority samples rather than learning general patterns. Can cause overfitting if the minority class is very small.


In [ ]:
np.random.seed(42)

def random_oversample_scratch(X, y, random_state=42):
    """Duplicate minority class samples until all classes have equal count.
    
    Returns a new (X, y) where every class has majority_count samples.
    The original samples are always included; only duplicates are added.
    """
    rng = np.random.default_rng(random_state)
    classes, counts = np.unique(y, return_counts=True)
    majority_count = counts.max()  # target size for every class

    X_parts, y_parts = [X], [y]   # start with originals

    for cls, count in zip(classes, counts):
        if count < majority_count:
            # Indices of all minority samples
            minority_idx = np.where(y == cls)[0]
            # Sample WITH replacement to fill the gap
            extra_idx = rng.choice(minority_idx, size=majority_count - count, replace=True)
            X_parts.append(X[extra_idx])
            y_parts.append(y[extra_idx])

    return np.vstack(X_parts), np.concatenate(y_parts)


X_over, y_over = random_oversample_scratch(X_train, y_train, random_state=42)

print('After oversampling:')
vals, cnts = np.unique(y_over, return_counts=True)
for v, c in zip(vals, cnts):
    print(f'  Class {v}: {c} samples')

# Fit logistic regression on the resampled data
lr_over = LogisticRegression(random_state=42, max_iter=500)
lr_over.fit(X_over, y_over)

y_pred_over = lr_over.predict(X_test)
y_prob_over = lr_over.predict_proba(X_test)[:, 1]
metrics_over = evaluate('LR + Oversampling', y_test, y_pred_over, y_prob_over)

print()
print(pd.DataFrame([metrics_base, metrics_weighted, metrics_over]).to_string(index=False))

## Strategy 3: Random Undersampling

**Idea:** randomly remove majority class samples until the classes are balanced.

Instead of adding minority copies, we discard majority samples. The model trains on less data overall but sees a balanced distribution.

**Pros:** speeds up training; no duplication; removes potentially redundant majority samples.  
**Cons:** discards real data — can hurt performance when the minority class is very small (because you end up with very few training samples total). Not recommended when data is scarce.


In [ ]:
np.random.seed(42)

def random_undersample_scratch(X, y, random_state=42):
    """Randomly remove majority class samples until all classes have equal count.
    
    Returns a new (X, y) where every class has minority_count samples.
    This loses data from the majority class.
    """
    rng = np.random.default_rng(random_state)
    classes, counts = np.unique(y, return_counts=True)
    minority_count = counts.min()  # target size for every class

    X_parts, y_parts = [], []

    for cls in classes:
        cls_idx = np.where(y == cls)[0]  # all indices for this class
        # Sample WITHOUT replacement — keep minority_count from each class
        sampled_idx = rng.choice(cls_idx, size=minority_count, replace=False)
        X_parts.append(X[sampled_idx])
        y_parts.append(y[sampled_idx])

    return np.vstack(X_parts), np.concatenate(y_parts)


X_under, y_under = random_undersample_scratch(X_train, y_train, random_state=42)

print('After undersampling:')
vals, cnts = np.unique(y_under, return_counts=True)
for v, c in zip(vals, cnts):
    print(f'  Class {v}: {c} samples')
print(f'  Total samples used: {len(y_under)} (down from {len(y_train)})')

lr_under = LogisticRegression(random_state=42, max_iter=500)
lr_under.fit(X_under, y_under)

y_pred_under = lr_under.predict(X_test)
y_prob_under = lr_under.predict_proba(X_test)[:, 1]
metrics_under = evaluate('LR + Undersampling', y_test, y_pred_under, y_prob_under)

print()
print(pd.DataFrame([metrics_base, metrics_over, metrics_under]).to_string(index=False))

## SMOTE: Synthetic Minority Oversampling Technique

Random oversampling duplicates existing minority samples. SMOTE generates **new synthetic samples** instead.

**How it works:**

1. For each minority sample $\mathbf{x}_i$, find its $k$ nearest minority-class neighbours.
2. Randomly pick one of those neighbours, say $\mathbf{x}_{nn}$.
3. Generate a synthetic sample anywhere on the **line segment** between $\mathbf{x}_i$ and $\mathbf{x}_{nn}$:

$$\mathbf{x}_{syn} = \mathbf{x}_i + \lambda \cdot (\mathbf{x}_{nn} - \mathbf{x}_i), \quad \lambda \sim \text{Uniform}(0, 1)$$

The synthetic samples fill in the **neighbourhood of the minority class** rather than simply copying existing points. The model is exposed to more of the minority manifold and learns its shape more reliably.

**Pros:** generates genuinely new samples; less overfitting than pure duplication; widely used.  
**Cons:** can introduce noise if minority class is very tight or noisy; doesn't work well in very high dimensions; still a form of data augmentation (apply only to training data).


In [ ]:
np.random.seed(42)

from sklearn.neighbors import NearestNeighbors

def smote_scratch(X, y, k_neighbors=5, random_state=42):
    """SMOTE: generate synthetic minority samples by interpolating between neighbours.
    
    For each synthetic sample needed:
      1. Pick a random minority sample x_i.
      2. Pick a random neighbour x_nn from x_i's k nearest minority neighbours.
      3. New sample = x_i + lambda * (x_nn - x_i), lambda ~ Uniform(0,1).
    """
    rng = np.random.default_rng(random_state)

    classes, counts = np.unique(y, return_counts=True)
    majority_count  = counts.max()
    minority_class  = classes[np.argmin(counts)]   # the label with fewer samples

    # Pull out the minority samples
    X_min = X[y == minority_class]
    n_to_generate = majority_count - len(X_min)    # how many synthetic samples we need

    # Fit kNN model ON minority samples only (k+1 because kNN includes the point itself)
    nbrs = NearestNeighbors(n_neighbors=k_neighbors + 1, algorithm='auto')
    nbrs.fit(X_min)
    _, indices = nbrs.kneighbors(X_min)  # indices shape: (n_min, k+1)
    # indices[:,0] is the sample itself; indices[:,1:] are the k actual neighbours

    synthetic = []
    for _ in range(n_to_generate):
        # Pick a random minority sample as the anchor
        i = rng.integers(len(X_min))
        # Pick one of its k nearest neighbours (skip index 0 = itself)
        nn_pos = rng.integers(1, k_neighbors + 1)
        nn_i   = indices[i, nn_pos]
        # Interpolation coefficient — anywhere along the line segment
        lam = rng.random()
        x_syn = X_min[i] + lam * (X_min[nn_i] - X_min[i])
        synthetic.append(x_syn)

    X_syn = np.array(synthetic)
    y_syn = np.full(n_to_generate, minority_class, dtype=y.dtype)

    # Combine original data with synthetic minority samples
    return np.vstack([X, X_syn]), np.concatenate([y, y_syn])


X_smote, y_smote = smote_scratch(X_train, y_train, k_neighbors=5, random_state=42)

print('After SMOTE:')
vals, cnts = np.unique(y_smote, return_counts=True)
for v, c in zip(vals, cnts):
    label = 'luxury' if v == 1 else 'regular'
    print(f'  Class {v} ({label}): {c} samples')
print(f'  Synthetic minority samples created: {cnts[vals == 1][0] - (y_train == 1).sum()}')

In [ ]:
np.random.seed(42)

# ── Visualise SMOTE in 2D (PCA projection) ────────────────────────────────────
pca = PCA(n_components=2, random_state=42)

# Fit PCA on original training data only
X_train_2d = pca.fit_transform(X_train)

# Project the synthetic minority samples using the same PCA
n_orig_minority = (y_train == 1).sum()            # original minority count
X_smote_2d = pca.transform(X_smote)              # project all SMOTE samples
y_smote_proj = y_smote                            # labels align

# Split into original vs. synthetic
orig_maj_idx  = np.where(y_train == 0)[0]         # original majority
orig_min_idx  = np.where(y_train == 1)[0]         # original minority
# Synthetic samples are appended at the end of X_smote
syn_idx = np.arange(len(X_train), len(X_smote))  # indices of synthetic samples

fig, ax = plt.subplots(figsize=(9, 6))

# Plot majority class (grey, small, behind)
ax.scatter(
    X_smote_2d[orig_maj_idx, 0], X_smote_2d[orig_maj_idx, 1],
    c='lightgrey', s=15, alpha=0.5, label='Majority (regular)', zorder=1
)
# Plot synthetic minority (cyan, medium)
ax.scatter(
    X_smote_2d[syn_idx, 0], X_smote_2d[syn_idx, 1],
    c='cyan', s=30, alpha=0.6, edgecolors='steelblue', linewidths=0.5,
    label='Synthetic minority (SMOTE)', zorder=2
)
# Plot original minority (blue, larger, on top)
ax.scatter(
    X_smote_2d[orig_min_idx, 0], X_smote_2d[orig_min_idx, 1],
    c='steelblue', s=60, edgecolors='navy', linewidths=0.8,
    label='Original minority (luxury)', zorder=3
)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('SMOTE: Synthetic minority samples fill in the minority region\n(PCA projection to 2D)')
ax.legend()
plt.tight_layout()
plt.show()

print('Observation: synthetic samples (cyan) fill the gaps between original minority samples (blue).')
print('They do NOT appear at random locations — they stay near the minority class manifold.')

In [ ]:
np.random.seed(42)

# ── Fit LR on SMOTE data and collect metrics ──────────────────────────────────
lr_smote = LogisticRegression(random_state=42, max_iter=500)
lr_smote.fit(X_smote, y_smote)

y_pred_smote = lr_smote.predict(X_test)
y_prob_smote = lr_smote.predict_proba(X_test)[:, 1]
metrics_smote = evaluate('LR + SMOTE (scratch)', y_test, y_pred_smote, y_prob_smote)

# ── Summary table: all strategies ────────────────────────────────────────────
all_metrics = [
    metrics_naive,
    metrics_base,
    metrics_weighted,
    metrics_over,
    metrics_under,
    metrics_smote,
]

summary_df = pd.DataFrame(all_metrics)
print('=== Strategy Comparison (test set) ===')
print(summary_df.to_string(index=False))

# Visual: F1 and AUC side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
methods = summary_df['Method']

for ax, metric, color in zip(axes, ['F1', 'AUC-ROC'], ['steelblue', 'mediumseagreen']):
    ax.barh(methods, summary_df[metric], color=color, alpha=0.85)
    ax.set_xlabel(metric)
    ax.set_title(f'{metric} by strategy (higher is better)')
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

## Strategy 4: Threshold Tuning

Logistic regression outputs a **probability**, and by default predicts class 1 when $P(y=1 | \mathbf{x}) > 0.5$.

The 0.5 threshold is arbitrary. On an imbalanced dataset the model is biased toward the majority class, so many true luxury listings get probabilities like 0.2–0.4 — below the default threshold but clearly non-trivial.

**Lowering the threshold** (e.g., to 0.2) means we flag more listings as luxury.  
- Recall goes up: we catch more true luxury listings.  
- Precision goes down: we also falsely flag more regular listings.

The **F1-maximising threshold** is the sweet spot that balances precision and recall. Finding it requires sweeping over thresholds and evaluating each one.

**Pros:** zero training cost — just post-process the probability output.  
**Best use case:** when you already have a well-calibrated model and just want to tune the operating point.


In [ ]:
np.random.seed(42)

from sklearn.metrics import precision_recall_curve

# Use the class-weighted model's probabilities for threshold tuning
# (well-calibrated starting point)
thresholds_sweep = np.linspace(0.01, 0.99, 200)  # sweep from 1% to 99%

precisions, recalls, f1s = [], [], []

for thresh in thresholds_sweep:
    preds = (y_prob_weighted >= thresh).astype(int)  # apply current threshold
    precisions.append(precision_score(y_test, preds, zero_division=0))
    recalls.append(recall_score(y_test, preds, zero_division=0))
    f1s.append(f1_score(y_test, preds, zero_division=0))

# Find the threshold that maximises F1
best_idx   = np.argmax(f1s)
best_thresh = thresholds_sweep[best_idx]
best_f1    = f1s[best_idx]

# ── Plot precision, recall, F1 vs threshold ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: precision, recall, F1 vs threshold
axes[0].plot(thresholds_sweep, precisions, label='Precision', color='steelblue')
axes[0].plot(thresholds_sweep, recalls,    label='Recall',    color='tomato')
axes[0].plot(thresholds_sweep, f1s,        label='F1',        color='purple', linewidth=2)
axes[0].axvline(best_thresh, color='black', linestyle='--',
                label=f'Best threshold = {best_thresh:.2f}')
axes[0].axvline(0.5, color='grey', linestyle=':', label='Default threshold = 0.50')
axes[0].set_xlabel('Decision threshold')
axes[0].set_ylabel('Score')
axes[0].set_title('Precision / Recall / F1 vs Threshold')
axes[0].legend(fontsize=8)

# Right: precision-recall curve with optimal point marked
pr_precision, pr_recall, pr_thresholds = precision_recall_curve(y_test, y_prob_weighted)
axes[1].plot(pr_recall, pr_precision, color='steelblue', linewidth=2, label='PR curve')
axes[1].scatter(
    recalls[best_idx], precisions[best_idx],
    color='red', zorder=5, s=100, label=f'Optimal F1 threshold = {best_thresh:.2f}'
)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Metrics at the optimal threshold
y_pred_thresh = (y_prob_weighted >= best_thresh).astype(int)
metrics_thresh = evaluate(f'LR (threshold={best_thresh:.2f})', y_test, y_pred_thresh, y_prob_weighted)

print(f'Optimal threshold: {best_thresh:.2f}  (default was 0.50)')
print(f'Best F1 at optimal threshold: {best_f1:.3f}')
print()
print(pd.DataFrame([metrics_base, metrics_weighted, metrics_thresh]).to_string(index=False))

In [ ]:
np.random.seed(42)

# ── Optional: compare with imbalanced-learn SMOTE ─────────────────────────────
try:
    from imblearn.over_sampling import SMOTE

    sm = SMOTE(random_state=42, k_neighbors=5)
    X_sm_ib, y_sm_ib = sm.fit_resample(X_train, y_train)

    lr_smote_ib = LogisticRegression(random_state=42, max_iter=500)
    lr_smote_ib.fit(X_sm_ib, y_sm_ib)

    y_pred_ib = lr_smote_ib.predict(X_test)
    y_prob_ib = lr_smote_ib.predict_proba(X_test)[:, 1]
    m_ib = evaluate('LR + SMOTE (imblearn)', y_test, y_pred_ib, y_prob_ib)

    print('imbalanced-learn SMOTE available.\n')
    print(pd.DataFrame([metrics_smote, m_ib]).to_string(index=False))
    print('\nResults should be very close — both implement the same SMOTE algorithm.')

except ImportError:
    print('imbalanced-learn not installed — using scratch SMOTE (validated above).')
    print('Install with: pip install imbalanced-learn')
    print(f'\nScratch SMOTE F1: {metrics_smote["F1"]:.3f}  AUC: {metrics_smote["AUC-ROC"]:.3f}')

## When NOT to Resample

Resampling is a training-time technique. It should **never** be applied to the test set.

**Why:** your test set represents the real-world distribution. If luxury listings are truly 5% of the market, your evaluation should reflect that. Resampling the test set would give you an optimistic, misleading picture of real-world performance.

**When resampling may not help or may hurt:**

1. **When test distribution is also imbalanced:** your model will be evaluated in a world where 95% of listings are regular. Forcing the model to see 50/50 during training creates a mismatch. Class weights are often a better fit.

2. **When the minority class is too small for SMOTE:** if you have fewer than $k+1$ minority samples, SMOTE cannot find $k$ neighbours. With very few samples, even interpolation produces unrealistic synthetic points.

3. **When features are mostly categorical:** SMOTE interpolates in feature space. Synthetic points between two listings in a distance-based space can produce non-sensical values for integer or categorical features.

4. **When you already have calibrated probabilities:** if you care about probability calibration (e.g., for pricing models), resampling changes the class prior and will mis-calibrate your probabilities. Use class weights or threshold tuning instead.

**Practical recommendation:** try class weights first (free). If that's not enough, try SMOTE on the training set. Always keep the test set untouched.


## Self-Check Questions

**Q1.** Your fraud detection model achieves 98.5% accuracy on the test set, but your manager is unhappy with the results. What is likely wrong, and what metric would you report instead?

<details><summary>Show answer</summary>

If fraud is ~1.5% of transactions, a model predicting "not fraud" for everything achieves 98.5% accuracy. The model is likely ignoring the minority class entirely. **Recall for the fraud class** is probably close to 0%.

Report **AUC-ROC, F1 for the fraud class, and the confusion matrix** instead. These metrics reveal whether the model is actually detecting fraud or just predicting the majority class.

</details>

---

**Q2.** You apply SMOTE to your full dataset before splitting into train and test sets. Why is this a mistake?

<details><summary>Show answer</summary>

This is **data leakage**. SMOTE generates synthetic samples by interpolating between real minority samples. If some of those real samples end up in the test set, the synthetic training samples are effectively derived from test-set data.

The model gets an indirect "preview" of the test distribution during training, inflating test performance estimates.

**Correct workflow:** split first, then apply SMOTE only to the **training set**. The test set is never touched.

</details>

---

**Q3.** Class weights, oversampling, undersampling, and SMOTE all address class imbalance. Describe the key trade-off between oversampling and undersampling, and when you would choose one over the other.

<details><summary>Show answer</summary>

**Oversampling** duplicates minority samples → keeps all majority-class information, but risks overfitting to specific minority examples (especially when the minority class is tiny).

**Undersampling** removes majority samples → faster training and no overfitting from duplicates, but discards potentially useful majority data. When the majority class is very large and the minority class is small, undersampling can leave you with very few training samples overall, degrading general model performance.

**Prefer oversampling (or SMOTE)** when data is limited and every sample is valuable.  
**Prefer undersampling** when the majority class is enormous and training speed matters.  
**Prefer class weights** when data volume is fine and you want the simplest possible solution — it's zero cost and often just as effective.

</details>


## Key Takeaways

1. **Accuracy is a trap on imbalanced data.** A model predicting the majority class for every sample can achieve very high accuracy while being completely useless. Always evaluate with **AUC-ROC, F1, precision, and recall** for the minority class.

2. **Try class weights first — zero cost.** `class_weight='balanced'` in sklearn upweights the minority class in the loss function with a single parameter change. It's often surprisingly effective.

3. **SMOTE when weights aren't enough.** It generates synthetic minority samples in the feature space rather than duplicating existing ones, giving the model more minority-class diversity to learn from.

4. **Never resample the test set.** The test set must reflect the real-world class distribution. Resampling it produces optimistic, misleading evaluation numbers.

5. **Threshold tuning is a free post-processing step.** After training, sweep the decision threshold and choose the value that maximises F1 (or whatever metric your business cares about). The default 0.5 is rarely optimal for imbalanced problems.

6. **SMOTE before train/test split = data leakage.** Always split first, then apply SMOTE only to training data.
